# 00 · A tour of `posydon-grid`

This series of notebooks takes apart how `posydon-grid setup` turns one ini file into a run directory of MESA inlists, sources and scripts. Use it to get a feel for the code, to try changes, and as a starting point for your own experiments.

| Notebook | What it covers | Main modules |
|---|---|---|
| **00 · Tour** (this one) | the whole thing from the command line | `posydon/CLI/grid/main.py` |
| **01 · Inlist files** | reading and writing MESA inlists | `inlist.py` |
| **02 · Layers** | how four sources are stacked into one inlist | `layers.py` |
| **03 · Ini and grid types** | the new ini format, the grid-type registry, errors | `config.py`, `grid_types.py`, `template.py` |
| **04 · Releases and reruns** | the inlist repository: manifests, reruns, what a variant changes | `manifest.py`, `resolve.py` |
| **05 · Building inlists** | MESA defaults, generated keys, ZAMS models, formation steps | `defaults.py`, `build.py`, `explain.py` |
| **06 · Extras and column lists** | Fortran sources, makefiles, `mk`, column lists | `extras.py` |
| **07 · Submission and record** | the `posydon-run-grid` command line, shell/SLURM scripts, `setup_record.ini` | `submit.py`, `record.py` |
| **08 · Step by step, and legacy inis** | `prepare` / `explain_grid` / `setup_grid`, comparing runs, old ini files | `steps.py`, `legacy.py` |

Every notebook runs on its own, in a few seconds, without MESA: `nb_env.py` points `$MESA_DIR` at a stub MESA and writes everything into `_work/` next to the notebooks.

**You need** a checkout of [POSYDON-MESA-INLISTS](https://github.com/POSYDON-code/POSYDON-MESA-INLISTS) on the `v2_sources` branch. Set `POSYDON_INLIST_REPO` to it (or keep it at `~/.posydon_mesa_inlists`). The notebooks only read it.

In [1]:
import os
# os.environ["POSYDON_INLIST_REPO"] = "/path/to/POSYDON-MESA-INLISTS"   # if not set already

from nb_env import env, fresh, write, show, tree, grid_csv, cli

ENV = env()
print("POSYDON         ", ENV.root)
print("inlist repo     ", ENV.inlists)
print("MESA_DIR (stub) ", ENV.mesa)
print("work folder     ", ENV.work)

POSYDON          /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a
inlist repo      /Users/max/Documents/POSYDON-MESA-INLISTS-v2-base
MESA_DIR (stub)  /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/mesa_stub
work folder      /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work


## The big picture

A setup runs eight steps (`posydon/grids/setup/steps.py`). Steps 1–3 only read and check; nothing is written until all of them pass.

| Step | What happens | Code |
|---|---|---|
| 1 | read and check the ini | `config.read_config` (new format) or `legacy.read_legacy_config` |
| 2 | read the grid points | `steps.read_grid_points` |
| 3 | find the scenario's inlists; fill in MESA's defaults | `resolve.resolve_scenario`, `defaults.fill_defaults` |
| 4 | copy Fortran sources and makefiles, compile | `extras.stage_extras`, `compile_executables` |
| 5 | stack the layers and write the inlists | `build.build_inlists` |
| 6 | copy the column lists | `extras.stage_column_lists` |
| 7 | write the `posydon-run-grid` command and its scripts | `submit.command_line`, `write_scripts` |
| 8 | record what it was set up from | `record.write_record` |

Every inlist value comes from one of four **layers**; a higher layer wins key by key:

```
4. generated         keys the setup writes itself (wiring, output, ZAMS model)
3. user_inlists      your own files
2. posydon_inlists   POSYDON's physics: the scenario's release (+ rerun)
1. mesa_inlists      MESA's own defaults
```

## The commands

`posydon-grid` has five subcommands. `posydon-setup-grid` still works but is deprecated.

In [2]:
cli("--help")

usage: posydon-grid [-h] {types,init,explain,setup,run} ...

Set up and run grids of MESA runs for POSYDON.

options:
  -h, --help            show this help message and exit

commands:
  {types,init,explain,setup,run}
    types               list the grid types
    init                write a starter ini file for a grid type
    explain             show where each inlist value comes from; write nothing
    setup               write the run directory of a grid
    run                 run the grid (all arguments go to posydon-run-grid
                        unchanged)


0

In [3]:
cli("types")

HMS-HMS       two hydrogen-rich main-sequence stars
CO-HMS_RLO    a compact object and a hydrogen-rich main-sequence star
CO-HeMS       a compact object and a helium main-sequence star
HeMS-HMS      a helium and a hydrogen-rich main-sequence star
single_HMS    single hydrogen-rich main-sequence stars
single_HeMS   single helium main-sequence stars


0

## 1. Start an ini file

`posydon-grid init <grid type> <file>` writes a starter ini for a grid type. It fills in what the grid type decides and leaves what only you know as `<...>` placeholders.

In [4]:
tour = fresh("00_tour")
cli("init", "CO-HeMS", tour / "starter.ini")
show(tour / "starter.ini")

Wrote /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/00_tour/starter.ini; fill in the <...> values.
; posydon-grid settings for a CO-HeMS grid:
; a compact object and a helium main-sequence star.
; Written by posydon-grid init CO-HeMS. Paths are relative to this file.
; Check with: posydon-grid explain <this file>
; Set up with: posydon-grid setup <this file>

[grid]
type = CO-HeMS
mode = fixed                ; fixed | dynamic (dynamic needs psycris_inifile)
points = grid.csv           ; the grid points, one row per MESA run
run_directory = .           ; where executables, inlists and output go

[scenario]                  ; which POSYDON inlists to use
release = v2.0
; rerun = matthias_lbv_thermo_dedt_noAgeLimit  ; a physics variant of v2.0
ref = v2_sources            ; branch or tag of POSYDON-MESA-INLISTS;
                            ; pin a production grid with commit = <hash>
; source = path             ; use in

## 2. Fill it in

For the notebooks we change three things in the starter:

- `source = path` and `inlist_repo`: use your checkout of the inlist repository as it is, without git (the starter uses `ref = v2_sources`, which clones and checks out `~/.posydon_mesa_inlists`);
- `backend = shell`: write a shell script instead of SLURM scripts, so no account is needed;
- `run_directory`: a folder of its own.

Paths in the ini are relative to the ini's folder.

In [5]:
grid_csv(tour)
ini = write(tour / "grid.ini", f"""
    [grid]
    type = CO-HeMS
    mode = fixed
    points = grid.csv
    run_directory = run

    [scenario]
    release = v2.0
    source = path
    inlist_repo = {ENV.inlists}

    [output]
    history_interval = 1
    star1 = history, final_model
    star2 = none
    binary = history

    [submission]
    backend = shell
    """)
show(ini)

[grid]
type = CO-HeMS
mode = fixed
points = grid.csv
run_directory = run

[scenario]
release = v2.0
source = path
inlist_repo = /Users/max/Documents/POSYDON-MESA-INLISTS-v2-base

[output]
history_interval = 1
star1 = history, final_model
star2 = none
binary = history

[submission]
backend = shell


## 3. Explain: where does each value come from?

`posydon-grid explain` runs steps 1–3 and prints, for every inlist the setup will write, each key whose value does *not* come from MESA's defaults: the layer and file that set it, and what it overrode. Nothing is written.

In [6]:
import subprocess, sys
text = subprocess.run([sys.executable, str(ENV.root / "bin" / "posydon-grid"), "explain", str(ini)],
                      capture_output=True, text=True).stdout
lines = text.splitlines()
print("\n".join(lines[:40]))
print(f"... {len(lines)} lines in all")

binary_controls
  history_interval = 1  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist_project
      over 5  [mesa_inlists] r11701/mesa/default/defaults/binary/binary_controls.defaults
  append_to_star_history = .false.  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist_project
      over .true.  [mesa_inlists] r11701/mesa/default/defaults/binary/binary_controls.defaults
  photo_interval = 0  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist_project
      over 50  [mesa_inlists] r11701/mesa/default/defaults/binary/binary_controls.defaults
  terminal_interval = 10  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist_project
      over 1  [mesa_inlists] r11701/mesa/default/defaults/binary/binary_controls.defaults
  write_header_frequency = 100  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist_project
      over 10  [mesa_inlists] r11701/mesa/default/defaults/binary/binary_controls.defaults
  fm = 5d-3  [posydon_inlists] r11701/releases/

Try `grep`-ing it for a key you care about:

In [7]:
print("\n".join(line for line in lines if "mesh_delta_coeff" in line or line and not line.startswith(" ")))

binary_controls
binary_job
star1_controls
  mesh_delta_coeff = 1.0d0  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist1  (same value as below)
star1_job
star2_controls
  mesh_delta_coeff = 1.0d0  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist2  (same value as below)
star2_job
star1_formation_step0_controls
  mesh_delta_coeff = 1.0d0  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist1  (same value as below)
star1_formation_step0_job
star1_formation_step1_controls
  mesh_delta_coeff = 1.0d0  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist1  (same value as below)
star1_formation_step1_job
Keys the setup generates (wiring, output options, ZAMS models) are added on top when the inlists are written.


## 4. Set it up

`posydon-grid setup` writes the run directory. With the stub MESA the compile step does nothing, so there are no real executables; everything else is as on the cluster.

In [8]:
cli("setup", ini)
tree(tour / "run")

binary/inlist1
binary/inlist2
binary/inlist_project
binary/make/  (1 files)
binary/src/  (3 files)
column_lists/binary_history_columns.list
column_lists/history_columns.list
column_lists/profile_columns.list
grid_command.sh
mk
setup_record.ini
star1/inlist_step0
star1/inlist_step1
star1/make/  (1 files)
star1/src/  (2 files)
star2/make/  (1 files)
star2/src/  (2 files)


What it wrote:

- `binary/inlist_project`, `binary/inlist1`, `binary/inlist2`: the binary and its two stars;
- `star1/inlist_step0`, `inlist_step1`: CO-HeMS builds star 1 (a helium star) in formation steps;
- `binary/src`, `star1/src`, …: the Fortran sources, and `mk`, which compiles them;
- `column_lists/`: what MESA writes to history and profiles;
- `grid_command.sh`: runs `posydon-run-grid` over the grid points;
- `setup_record.ini`: which ini, POSYDON version, inlist repository and commit this came from.

In [9]:
show(tour / "run" / "setup_record.ini")

; What this run directory was set up from; written by posydon-grid setup.

[setup]
inifile = /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/00_tour/grid.ini
ini_format = new
posydon_version = 2.2.4+3.ga271efb93
mesa_dir = /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/mesa_stub
grid_type = CO-HeMS
mode = fixed

[scenario]
source = path
release = v2.0
inlist_repo = /Users/max/Documents/POSYDON-MESA-INLISTS-v2-base
inlist_commit = 115de83c33bad5013cb09837fcc679d1d45e816b
mesa_version = r11701
mesa = default


In [10]:
show(tour / "run" / "grid_command.sh")

#!/bin/bash

export OMP_NUM_THREADS=1

export MESASDK_ROOT=/opt/mesasdk-stub
source $MESASDK_ROOT/bin/mesasdk_init.sh
export MESA_DIR=/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/mesa_stub

for SLURM_ARRAY_TASK_ID in 0 1 2 ; do python /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/bin/posydon-run-grid --mesa-grid /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/00_tour/grid.csv --mesa-binary-executable /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/00_tour/run/binary/binary --mesa-star1-executable /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/00_tour/run/star1/star --mesa-star2-executable /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_r

## 5. Things to try

- Change `type` to `HMS-HMS` and add `zams = <inlist repo>/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data` to `[grid]`; run `explain` and `setup` again.
- Add `rerun = matthias_lbv_thermo_dedt_noAgeLimit` to `[scenario]` and see which values change in `explain` (notebook 04 does this systematically).
- Add a `[user_inlists]` section with `star1 = my_inlist1` (a file with `&controls  mesh_delta_coeff = 0.8d0 /`); `explain` shows it on top.
- Break the ini on purpose (a typo in a key, `type = MS-MS`, `mode = dynamic` without `psycris_inifile`) and read the error: it names the section and key.